# 相关概念
## 1 Group Invariance（群不变性）与Group Equivariance（群等变性）

---

## 1.1. 什么是“群”（Group）？

在数学中，“群”是一组元素，配合一个“运算”，满足：

* 闭合性（Closure）
* 结合律（Associativity）
* 单位元（Identity）
* 逆元素（Inverse）

在三维空间里，我们常说的 **SE(3)** 群就是：

* 所有**三维空间中的刚性变换**（平移 + 旋转）的集合。
* SE(3) = SO(3)（旋转） + R³（平移）

---

## 1.2. 群不变性（Group Invariance）

### 定义：

一个函数 $f(x)$ 对群 G **不变**，表示对任意的群元素 $g \in G$，都有：

$$
f(g \cdot x) = f(x)
$$

也就是说，对输入做任何一次群变换（如旋转、平移），函数的输出**完全不变**。

### 举例：

* 图像分类网络：对一只猫的图片旋转、平移，预测结果仍应是“猫” → 分类模型应具有**不变性**。
* 分子能量预测模型：分子整体旋转或移动，其能量不应变 → 能量函数需要对 SE(3) **不变**。

---

## 1.3. 群等变性（Group Equivariance）

### 定义：

一个函数 $f(x)$ 对群 G **等变**，表示：

$$
f(g \cdot x) = g \cdot f(x)
$$

也就是说，**对输入做的变换，会被模型“同步”到输出上。**

### 举例：

* 分子中的每个原子坐标变换后，预测的力方向也应该相应变换 → 模型输出力向量时应具备 SE(3) 等变性。
* 图像中的位移，造成特征图也平移（CNN 的卷积核就是 translation-equivariant）。

---

## 1.4. 两者的区别

| 特性     | 不变性（Invariance） | 等变性（Equivariance） |
| ------ | --------------- | ----------------- |
| 输出是否变动 | 不变              | 按群变换同步变化          |
| 应用场景   | 分类、能量预测         | 关键点定位、力预测         |
| 数学表达   | $f(gx) = f(x)$  | $f(gx) = g(f(x))$ |

---

## 1.5. 与 SE(3) Diffusion 的关系

SE(3) diffusion 中建模的是原子或残基的三维坐标：

* 如果你生成的是**结构**（坐标）：需要 **SE(3)-等变性**
* 如果你只关心最终的**属性（能量、类别）**：需要 **SE(3)-不变性**
* Diffusion Model 的 denoiser 需要遵循**等变性**，否则会破坏三维几何结构的对称性。




## 2 SE(3)与SO(3)


## 一句话区分

* **SO(3)**：三维空间中的**旋转群**，不包含平移。
* **SE(3)**：三维空间中的**刚性变换群** = 旋转（SO(3)）+ 平移（R³）。

---

## 2.1. SO(3)：Special Orthogonal Group in 3D

### 定义：

SO(3) 是由所有 **保持三维欧氏空间长度和角度不变的旋转矩阵** 组成的集合。

数学上：

$$
SO(3) = \{ R \in \mathbb{R}^{3 \times 3} \ |\ R^\top R = I,\ \det(R) = +1 \}
$$

* 是一个**连续李群**（Lie group），维度为 **3**。
* 通常参数化为欧拉角、轴角、四元数、旋转矩阵。

### 示例：

* 把一个向量绕 Z 轴旋转 90°。
* 把分子的所有原子绕某个轴转动。

---

## 2.2. SE(3)：Special Euclidean Group in 3D

### 定义：

SE(3) 是所有 **保持刚性结构的三维变换（旋转 + 平移）** 的集合：

$$
SE(3) = \left\{ \begin{bmatrix} R & t \\ 0 & 1 \end{bmatrix} \Bigg| R \in SO(3),\ t \in \mathbb{R}^3 \right\}
$$

* 可以理解为组合变换：先旋转，再平移。
* 是一个 6 维的李群（3维旋转 + 3维平移）。

### 示例：

* 把整个分子先绕轴旋转，再整体移动一个向量。
* 机器人手臂的末端执行器的“位姿”（position + orientation）属于 SE(3)。

---

## 二者关系图示

```
SO(3)：     仅旋转
             🔄

SE(3)： 旋转 🔄 + 平移 ➡️
```

你可以认为：

> SO(3) 是 SE(3) 的一个子群（只含旋转部分）。

---

## 为什么在 Diffusion Model 中重要？

| 任务      | 使用哪个群？                | 为什么            |
| ------- | --------------------- | -------------- |
| 分子能量预测  | SE(3)-**invariant**   | 结构变动不应改变能量     |
| 三维结构生成  | SE(3)-**equivariant** | 结构在任何姿态下都能合理生成 |
| 蛋白质残基定位 | SE(3)-equivariance    | 模型需要学习三维空间结构信息 |

很多结构生成模型（如 **GeoDiff**, **EDM**, **FrameDiff**, **SE(3)-Diffusion**）都要求网络在 SE(3) 群下是等变（equivariant）的，这样才能保证生成结构在三维空间中具有物理意义。

---

### 总结一句话：

| 群         | 含义      | 维度 | 用途       |
| --------- | ------- | -- | -------- |
| **SO(3)** | 仅旋转     | 3  | 旋转建模     |
| **SE(3)** | 旋转 + 平移 | 6  | 三维刚性变换建模 |




## 3 SE(3)与$SE(3)^N$的联系和区别

---

##  3.1 什么是 **SE(3)**？

**SE(3)** 是一个**李群（Lie group）**，表示 **三维欧几里得空间中的刚体变换群**，也就是 **“旋转 + 平移”** 的组合。

* **S**pecial：表示旋转矩阵是正交且行列式为 1 的；
* **E**uclidean：欧几里得空间；
* **3**：三维空间；
* 所以 **SE(3)** 包含所有能在三维空间中做“刚体变换”的操作（不改变形状或大小）。

每个 **SE(3)** 的元素是一个旋转 + 平移：

$$
g = (R, t), \quad R \in SO(3),\ t \in \mathbb{R}^3
$$

可以表示为 4×4 的齐次变换矩阵：

$$
\begin{bmatrix}
R & t \\
0 & 1
\end{bmatrix}
$$

---

## 3.2 那么，什么是 **$SE(3)^{N}$**？

**$SE(3)^{N}$** 表示的是 **N 个独立的 SE(3) 元素**组成的笛卡尔积：

$$
SE(3)^N = \underbrace{SE(3) \times SE(3) \times \cdots \times SE(3)}_{N\ \text{times}}
$$

也就是说，它是一个 **N 维刚体配置空间**，每个元素都描述一个刚体的姿态（旋转 + 平移）。

---

## 3.3 举个蛋白质的例子：

假设你有一个含有 **N 个残基**的蛋白质主链，每个残基的位置与朝向都可以用一个 **SE(3)** 元素表示（即一个“刚性坐标系”或“frame”）。

那整个主链的构象就可以表示为：

$$
(G_1, G_2, ..., G_N) \in SE(3)^N
$$

也就是你在 **SE(3)** 空间中放了 **N 个刚体**，组成一个结构。

---

## 3.4 SE(3) vs $SE(3)^{N}$ 总结对比：

| 内容   | SE(3)                 | $SE(3)^{N}$ |
| ---- | --------------------- | ----------------- |
| 定义   | 三维空间中一个刚体的变换（旋转 + 平移） | N 个刚体的空间配置（蛋白质结构） |
| 数学结构 | 李群                    | 李群的笛卡尔积（N 次）      |
| 参数维度 | 6维（3旋转 + 3平移）         | 6×N 维             |
| 例子   | 一个残基的位置与朝向            | 整条蛋白质主链的构象        |

---


## 3.5 什么是笛卡尔积？

### 定义：

给定两个集合 $A$ 和 $B$，它们的 **笛卡尔积** $A \times B$ 是所有可能的 **有序对** $(a, b)$，其中 $a \in A$，$b \in B$。

$$
A \times B = \{(a, b) \mid a \in A,\ b \in B \}
$$

也可以推广到多个集合的积，比如：

$$
A \times B \times C = \{(a, b, c) \mid a \in A,\ b \in B,\ c \in C \}
$$

---

## 举个通俗例子

### 例 1：点的组合

设 $A = \{1, 2\}$，$B = \{x, y\}$，那么：

$$
A \times B = \{(1, x), (1, y), (2, x), (2, y)\}
$$

这个集合表示：A 中的每个元素和 B 中的每个元素**配对一次**。

---

### 例 2：二维空间

$$
\mathbb{R}^2 = \mathbb{R} \times \mathbb{R}
$$

这表示所有可能的二维实数点 $(x, y)$，x 和 y 都来自实数集合 $\mathbb{R}$。

类似地：

$$
\mathbb{R}^3 = \mathbb{R} \times \mathbb{R} \times \mathbb{R}
$$

---

## 回到你前面的问题：$SE(3)^N$ 是什么意思？

* 这是 $SE(3) \times SE(3) \times \cdots \times SE(3)$（总共 N 个）的笛卡尔积。
* 表示 N 个独立的 SE(3) 元素，每一个代表一个“刚体”（比如一个残基）的空间位置与方向。

所以：

> **$SE(3)^{N}$ 是 N 个 SE(3) 元素组成的有序元组，每个元素来自 SE(3)**。

---

##  总结：

| 概念                | 含义                                      |
| ----------------- | --------------------------------------- |
| 笛卡尔积 $A \times B$ | 所有可能的有序对 $(a, b)$，其中 $a \in A, b \in B$ |
| $\mathbb{R}^n$    | 实数集合 $\mathbb{R}$ 的 n 次笛卡尔积，即 n 维空间     |
| $SE(3)^N$         | N 个刚体位姿的组合空间，每个来自 SE(3)                 |



---

## 为什么使用 $SE(3)^{N}$？

* 因为蛋白质是多个残基组合的结构，每个残基在三维空间中都有自己的frame（刚体）。
* 所以蛋白质主链可以自然地表示为 **N 个 SE(3)** 元素组成的结构——即在 **$SE(3)^{N}$ 空间**上建模。
* 这使得我们可以用**李群结构**（如等变网络）进行有物理意义的学习和生成。





## 4. SE(3)等变网络？


## 4.1 一句话解释：

**SE(3) 等变神经网络**是一类对**三维空间中的旋转和平移（刚体变换）具有结构性响应**的神经网络，也就是说：

> 如果你对输入做一个 SE(3) 变换（旋转或平移），网络输出也会**相应地变换**，但**信息不会丢失或乱掉**。

---

## 4.2 等变 vs 不变：先理解基本概念

| 概念                  | 数学定义                 | 意思           | 举例                       |
| ------------------- | -------------------- | ------------ | ------------------------ |
| **不变（Invariant）**   | $f(T(x)) = f(x)$     | 输入变换了，输出不变   | 图像分类（把猫图像旋转一下，还是猫）       |
| **等变（Equivariant）** | $f(T(x)) = T'(f(x))$ | 输入变换了，输出也跟着变 | 3D蛋白质结构预测（旋转输入结构，输出也应旋转） |

---

## 4.3 什么是 SE(3)？

* **SE(3)** 是 **3D空间中的刚体变换群**：**旋转 + 平移**
* 一个 SE(3) 元素可以表示为 $g = (R, t)$，其中：

  * $R \in SO(3)$：3D旋转
  * $t \in \mathbb{R}^3$：平移向量

---

## 4.4 SE(3) 等变网络：动机

在很多任务中，比如：

* 蛋白质结构预测（AlphaFold）
* 分子建模与力场模拟（SchNet、DimeNet）
* 三维点云理解
* 药物对接

输入的是三维坐标表示的对象，**这些对象在空间中的朝向不应该影响预测结果的质量**。

所以我们需要构建一个网络，满足：

> 如果我们把输入分子旋转或平移一下，网络输出也应该“跟着转/移”，而不是输出完全不同的结果。

---

## 4.5 怎么实现 SE(3) 等变网络？

这类网络需要在以下层面都保持等变性：

| 模块   | 等变要求                     | 举例                      |
| ---- | ------------------------ | ----------------------- |
| 特征表示 | 用向量（1阶张量）、矩阵（2阶张量）表示空间特征 | 原子的位置、力向量               |
| 邻接关系 | 通过距离或相对位置建立              | 不依赖绝对坐标                 |
| 卷积操作 | 必须在局部坐标系中做旋转对齐           | 类似于 SE(3)-卷积            |
| 激活函数 | 对张量/向量设计可区分的激活方式         | 使用 Clebsch-Gordan 分解等技术 |

---

## 4.6 常见的 SE(3) 等变网络类型

| 网络名称                           | 特点                             | 应用                |
| ------------------------------ | ------------------------------ | ----------------- |
| **SE(3)-Transformer**          | 基于 transformer 架构，引入 SE(3) 对称性 | AlphaFold、蛋白质结构预测 |
| **Tensor Field Network (TFN)** | 使用张量表示，适合建模高阶空间关系              | 分子建模              |
| **E(n)-GNN / EGNN**            | 更简单、仅保留等变性，不追求高阶张量             | 分子/蛋白质预测、3D点云     |
| **DimeNet / SchNet**           | 面向分子动力学建模，基于消息传递               | 分子性质预测            |

---

## 4.7 举个实际例子帮助理解：

假设你有一个蛋白质残基的位置和方向，想预测这个点上的一个力向量：

* 如果你把整个蛋白质转了一圈（比如绕 z 轴旋转 90°）；
* 那么你希望预测出来的“力向量”也相应地跟着旋转；
* 如果你使用的是 SE(3) 等变网络，网络自然会保证这个力向量“跟着转”；
* 如果你使用普通的 MLP，它可能会输出完全不同的结果，因为它“看不懂”旋转结构之间的关系。

---

## 4.8 总结：

| 项目           | 解释                              |
| ------------ | ------------------------------- |
| SE(3) 等变神经网络 | 能对输入做刚体变换（旋转+平移）时，输出也“跟着动”的神经网络 |
| 应用场景         | 3D建模、蛋白质预测、药物设计、机器人视觉等          |
| 核心优势         | 建模时自动考虑空间对称性，更符合物理世界规律，具有更强泛化能力 |




## 5. Lie Group（李群）、Riemannian Manifold（黎曼流形）、SO(3)和SE(3)的含义、联系和区别


## 一、定义篇：每个概念是什么意思？

| 概念                            | 定义                                          | 举例                                                |
| ----------------------------- | ------------------------------------------- | ------------------------------------------------- |
| **Lie Group（李群）**             | 同时是**光滑流形**和**群**的数学结构。其“乘法”和“逆元”操作是光滑（可导）的 | SO(3)、SE(3)、实数加法群 $\mathbb{R}^n$                  |
| **Riemannian Manifold（黎曼流形）** | 在光滑流形每个点定义了**内积结构**（测角度、距离、梯度）              | 球面 $S^2$、欧几里得空间 $\mathbb{R}^3$、SO(3)、SE(3)（加上度量后） |
| **SO(3)**                     | **李群**，表示三维空间中的**旋转操作**（特殊正交群，3×3旋转矩阵）      | 描述物体的朝向                                           |
| **SE(3)**                     | **李群**，表示三维空间中的**刚体变换（旋转+平移）**              | 蛋白质残基的位置与方向、机器人手臂姿态                               |

---

## 二、联系篇：它们之间是什么关系？

1. **SO(3)、SE(3)** 都是具体的 **Lie Group**

   * SO(3) 是纯旋转（不动点）
   * SE(3) 是旋转+平移（刚体运动）

2. 所有 **Lie Group 都是光滑流形**，因此也属于 **Riemannian Manifold**（加上度量后）

3. 所以：

   ```
   SO(3) ⊂ SE(3) ⊂ Lie Group ⊂ Riemannian Manifold
   ```

4. **SE(3) = SO(3) × ℝ³** （数学上是半直积）

   * SO(3)：决定方向（旋转部分）
   * ℝ³：决定位置（平移部分）

---

## 三、区别篇：它们有何不同？

| 比较点          | Lie Group | Riemannian Manifold | SO(3) | SE(3)         |
| ------------ | --------- | ------------------- | ----- | ------------- |
| 是否是群结构       | ✅         | ❌（不是群）              | ✅     | ✅             |
| 是否是流形        | ✅         | ✅                   | ✅     | ✅             |
| 是否能定义距离/角度   | ✅（有度量）    | ✅                   | ✅     | ✅             |
| 是否代表物理空间中的操作 | 一般抽象      | 一般抽象                | 是（旋转） | 是（刚体运动）       |
| 维度           | 不固定       | 不固定                 | 3维流形  | 6维流形（3旋转+3平移） |

---

## 四、图解关系（简化）

```text
       Riemannian Manifold
               ↑
           Lie Group
            ↑     ↑
         SO(3)   SE(3)
```

* **SO(3)**：旋转构成的李群（3维）
* **SE(3)**：旋转+平移构成的李群（6维）
* 都属于**Lie Group**
* 所有李群都可以看作**带有额外结构的黎曼流形**

---

## 五、在蛋白质建模中的应用类比

| 结构           | 数学空间   | 含义            |
| ------------ | ------ | ------------- |
| 残基的位置        | ℝ³     | 原子坐标（平移）      |
| 残基的朝向        | SO(3)  | 局部坐标系的方向      |
| 残基的姿态（位置+方向） | SE(3)  | 整个刚体结构（Frame） |
| 所有残基组成的结构空间  | SE(3)ⁿ | 蛋白质主链的三维结构空间  |

---

## 六、总结一句话

> **SE(3)** 是三维刚体运动的空间，既有旋转（SO(3)）又有平移（ℝ³）；它是一个 **Lie Group**，也可以赋予 **Riemannian结构**。我们在蛋白质建模、生成模型中，需要在这个空间上建模，因为蛋白质的残基位置和方向共同决定结构，而不是单独的点。


